In [0]:
from pyspark.sql import functions as F

# Read Silver tables
shift = spark.table(
    "workforce_analytics.silver.fact_shift"
).alias("f")

people = spark.table(
    "workforce_analytics.silver.dim_people"
).alias("p")

county = spark.table(
    "workforce_analytics.silver.dim_county"
).alias("c")

# Join dimensions
fact_shift_gold = (
    shift
    .join(
        people,
        F.col("f.people_id") == F.col("p.people_id"),
        "left"
    )
    .join(
        county,
        F.col("f.county_id") == F.col("c.county_id"),
        "left"
    )
    .select(
        *[F.col(f"f.{col}") for col in shift.columns],

        F.col("p.employee_name"),
        F.col("p.job_role"),
        F.col("p.employment_type"),
        F.col("p.team"),

        F.col("c.county_name"),
        F.col("c.region"),
        F.col("c.country"),
        F.col("c.population_band"),
        F.col("c.urban_rural")
    )
)

# Save Gold table
(
    fact_shift_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workforce_analytics.gold.fact_shift_gold")
)

print("Gold Fact Shift created successfully.")

In [0]:
gold_shift = spark.table(
    "workforce_analytics.gold.fact_shift_gold"
)

print("Snapshot counts:")
display(
    gold_shift.groupBy("snapshot_date")
    .count()
    .orderBy("snapshot_date")
)

print("Total rows:", gold_shift.count())

print("Duplicate shift IDs:")
display(
    gold_shift.groupBy("snapshot_date", "shift_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Missing employee names:",
    gold_shift.filter(F.col("employee_name").isNull()).count()
)

print(
    "Missing county names:",
    gold_shift.filter(F.col("county_name").isNull()).count()
)